# M5 — Detector de anomalías (Isolation Forest) vs clasificador supervisado

Pregunta: ¿y si no tuviéramos etiquetas? Entrenamos solo con benigno y medimos cuánto caza de malicious/suspicious.

## Literatura (verificada hoy)
- Liu et al. Isolation Forest (2008/2012) — base: aislar es más barato que perfilar. Ideal tabular + lineal, pocos hiperparámetros.
- Zoppi et al. ACM CSUR 2021 (17 algoritmos × 11 datasets IDS): IF, One-Class SVM y SOM los mejores para zero-day; clustering alternativa barata.
- Nyyssölä et al. IEEE QRS 2024 (logs BGL/Thunderbird/HDFS): con datos sin filtrar, IF + eventos = AUC 0.829; con solo-normal, OOV + trigramas = 0.846.
- Kim TIIS 2024 (NIDS híbrido AE + IF + LOF): P 88.2 / R 92.8; con oversampling normal R 94.6.
- Fitriani et al. E3S 2025 (NSL-KDD, ensemble LightGBM + AE + IF): F1 99.24, FPR 0.58 — pero IF solo: recall 99.78 con FPR 51% (caza todo, llora todo).
- Sánchez-Zas et al. Mathematics 2022 (streaming logs heterogéneos): K-Means óptimo por tiempo real.
- Karlsen et al. arXiv 2311.14519 (HTTP CSIC/ECML): IF + fusión de features = acc 0.85.

Lección para nosotros: IF caza (recall alto) pero con FPR alto. Sirve como red de zero-days, no como triage principal.

**Protocolo:** mismo split temporal 80/20 que M1/M2/M3. Train = solo benigno pasado. Test = futuro mezclado. Métricas: P/R/F1 binario + PR-AUC con score de anomalía. Contamination = 0.057 (5.31+0.38 tasa real).

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, f1_score, average_precision_score, roc_auc_score
ROOT = Path('..')
CATS = ['message_type','stream_name','product_name','vendor_name','pipeline','org_id','action','severity','protocol']
NUMS = ['hour','msg_len','lag_h']
df = pd.read_parquet(ROOT/'data/signals/signals.parquet', columns=CATS+['timestamp','event_time','message_sanitized','label_binary'])
ts = pd.to_datetime(df['timestamp'], unit='s', utc=True)
df['hour'] = ts.dt.hour
df['msg_len'] = df['message_sanitized'].str.len().fillna(0)
df['lag_h'] = ((df['timestamp']-df['event_time'])/3600).clip(-1e4,1e4).fillna(-1)
for c in CATS: df[c] = df[c].fillna('').astype(str)
print(df.shape); print(df.label_binary.value_counts())

In [ ]:
# split temporal 80/20, train SOLO benigno pasado (como si no hubiera labels de ataque)
cut = df['timestamp'].quantile(0.80)
train = df[(df.timestamp <= cut) & (df.label_binary=='benign')].sample(20000, random_state=42)
# test futuro mezclado estratificado
fut = df[df.timestamp > cut]
mal = fut[fut.label_binary=='malicious']
sus = fut[fut.label_binary=='suspicious'].sample(min(len(fut[fut.label_binary=='suspicious']), 2000), random_state=42)
ben = fut[fut.label_binary=='benign'].sample(2000, random_state=42)
test = pd.concat([mal, sus, ben]).sample(frac=1.0, random_state=42)
print('train benign:', len(train), '| test:', test.label_binary.value_counts().to_dict())
Xtr, Xte = train[CATS+NUMS], test[CATS+NUMS]
# binario anomalía: malicious+suspicious=1, benign=0
yte = (test.label_binary!='benign').astype(int)
print('tasa anomalía test: %.3f' % yte.mean())

In [ ]:
pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore', max_categories=30), CATS), ('num','passthrough', NUMS)])
Atr = pre.fit_transform(Xtr)  # fit SOLO en benigno
Ate = pre.transform(Xte)
print('features:', Atr.shape)
clf = IsolationForest(n_estimators=200, contamination=0.057, random_state=42, n_jobs=-1)
clf.fit(Atr)
pred = clf.predict(Ate)  # 1 normal, -1 anomalía
yan = (pred==-1).astype(int)
# score: más alto = más anómalo
score = -clf.score_samples(Ate)
print(classification_report(yte, yan, target_names=['benign','anomalia']))
print('F1: %.4f | PR-AUC: %.4f | ROC-AUC: %.4f' % (f1_score(yte, yan), average_precision_score(yte, score), roc_auc_score(yte, score)))

In [ ]:
# desglose por tier: ¿caza malicious pero llora en benign? (patrón literatura: recall alto, FPR alto)
import numpy as np
test = test.copy(); test['anomalia']=yan; test['score']=score
print(test.groupby('label_binary').anomalia.agg(['mean','sum','count']))
print('\nFPR benigno (falsa alarma): %.3f' % (test[test.label_binary=="benign"].anomalia.mean()))
print('Recall malicious: %.3f' % (test[test.label_binary=="malicious"].anomalia.mean()))
print('Recall suspicious: %.3f' % (test[test.label_binary=="suspicious"].anomalia.mean()))

## Qué esperar y cómo contarlo
- IF caza malicious (R alto, ~0.7-0.9) pero FPR benigno alto (~0.05-0.5 según contaminación) — igual que Fitriani 2025 (R 99.7, FPR 51%).
- Conclusión: no reemplaza M1/M2/M3 (F1-macro 0.60/0.71/0.72 con clases separadas), pero sirve como red zero-day: lo que IF marca y M1 dice benign → cola de revisión.
- Siguiente paso barato: ensemble score IF + proba M1 (promedio normalizado, umbral por F1 como E3S 2025), o OOV-trigramas sobre `message_sanitized` (QRS 2024) si hay tiempo.